## Run evaluation on different action policies, e.g. VLA

In [1]:
from VLABench.evaluation.evaluator import Evaluator
from VLABench.evaluation.model.policy.openvla import OpenVLA
from VLABench.evaluation.model.policy.base import RandomPolicy
from VLABench.tasks import *
from VLABench.robots import *
from huggingface_hub import snapshot_download
from pathlib import Path

demo_tasks = ["select_fruit"]
unseen = True
save_dir = "/home/jadelynn/VLABench/logs"

"""
hf_repo_id = "VLABench/pi0-fast-primitive"
hf_snapshot_dir = snapshot_download(repo_id=hf_repo_id, repo_type="model")
snapshot_path = Path(hf_snapshot_dir)

model_ckpt = next((str(p) for p in snapshot_path.rglob("openvla-7b") if p.is_dir()), None)
if model_ckpt is None:
    raise FileNotFoundError(f"Could not locate 'openvla-7b' within the Hugging Face snapshot at {snapshot_path}")
lora_ckpt = next((str(p) for p in snapshot_path.rglob("select_fruit+CSv1+lora") if p.is_dir()), None)
if lora_ckpt is None:
    raise FileNotFoundError(f"Could not locate 'select_fruit+CSv1+lora' within the Hugging Face snapshot at {snapshot_path}")
"""

/home/jadelynn/miniconda3/envs/vlabench/lib/python3.10/site-packages/dash/_jupyter.py:30: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  _dash_comm = Comm(target_name="dash")


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


'\nhf_repo_id = "VLABench/pi0-fast-primitive"\nhf_snapshot_dir = snapshot_download(repo_id=hf_repo_id, repo_type="model")\nsnapshot_path = Path(hf_snapshot_dir)\n\nmodel_ckpt = next((str(p) for p in snapshot_path.rglob("openvla-7b") if p.is_dir()), None)\nif model_ckpt is None:\n    raise FileNotFoundError(f"Could not locate \'openvla-7b\' within the Hugging Face snapshot at {snapshot_path}")\nlora_ckpt = next((str(p) for p in snapshot_path.rglob("select_fruit+CSv1+lora") if p.is_dir()), None)\nif lora_ckpt is None:\n    raise FileNotFoundError(f"Could not locate \'select_fruit+CSv1+lora\' within the Hugging Face snapshot at {snapshot_path}")\n'

In [2]:
import os
os.environ["MUJOCO_GL"] = "egl"

### Init evaluator

In [3]:
evaluator = Evaluator(
    tasks=demo_tasks,
    n_episodes=2,
    max_substeps=10,   
    save_dir=save_dir,
    visulization=True
)

Load the task episodes by seeds, instead of episodes


### Load basic random policy

In [4]:
random_policy = RandomPolicy(model=None)
result = evaluator.evaluate(random_policy)

Evaluating select_fruit of RandomPolicy: 100%|██████████| 2/2 [01:53<00:00, 56.87s/it]


### Load policies, take OpenVLA as example

In [6]:
# Load OpenVLA model
model_ckpt = "/home/jadelynn/openvla-lora/"
lora_ckpt = "/home/jadelynn/openvla-lora/"

In [7]:
policy = OpenVLA(
    model_ckpt=model_ckpt,
    lora_ckpt=lora_ckpt,
    norm_config_file= "/home/jadelynn/openvla-lora/config.json" #os.path.join(os.getenv("VLABENCH_ROOT"), "configs/model/openvla_config.json")
)

result = evaluator.evaluate(policy)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

ValueError: Can't find 'adapter_config.json' at '/home/jadelynn/openvla-lora/'

## Run evaluation on different VLMs

In [8]:
from VLABench.evaluation.model.vlm import *
from VLABench.evaluation.evaluator import VLMEvaluator

vlm_name = "GPT_4v" # valid names: ["GPT_4v", "Qwen2_VL", "InternVL2", "MiniCPM_V2_6", "GLM4v", "Llava_NeXT"]
fewshot_num = 0
task_list = ["mesh_and_texture/select_fruit"]

def initialize_model(model_name, *args, **kwargs):
    cls = globals().get(model_name)
    if cls is None:
        raise ValueError(f"Model '{model_name}' not found in the current namespace.")
    
    return cls(*args, **kwargs)


In [9]:
vlm = initialize_model(vlm_name)
evaluator = VLMEvaluator(
    tasks=task_list,
    n_episodes=2,
    data_path=os.path.join(os.getenv("VLABENCH_ROOT"), "../dataset", "vlm"),
    save_path=os.path.join(os.getenv("VLABENCH_ROOT"), "../logs/vlm"),
)

evaluator.evaluate(vlm, few_shot_num=fewshot_num)
result=evaluator.get_final_score_dict(vlm_name)


Load the task episodes by seeds, instead of episodes


AssertionError: Data path does not exist